# Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn transformers torch torchvision Pillow python-multipart omegaconf

# Create Config File

In [ ]:
yaml_config =  """
               model_path: "microsoft/table-transformer-detection"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# Build Model

In [ ]:
import torch
import io
from PIL import Image
from omegaconf import OmegaConf
from transformers import AutoImageProcessor, AutoModelForObjectDetection


class TableDetection:
  def __init__(self, config_path):
    self.config = OmegaConf.load(config_path)
    self.processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-detection")
    self.model = AutoModelForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

  def __call__(self, imageBytes):
    # chuyển ảnh về dạng RGB
    image = Image.open(io.BytesIO(imageBytes)).convert("RGB")
    # Tiền xử lý
    input = self.processor(images = image, return_tensors = "pt")
    with torch.no_grad():
        output = self.model(**input)

    target_sizes = torch.tensor([image.size[::-1]])
    results = self.processor.post_process_object_detection(
                output, threshold=0.9, target_sizes=target_sizes
            )[0]

    # Trích xuất kết quả ra list dictionary
    detected_tables = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
      detected_tables.append({
        "label": self.model.config.id2label[label.item()],
        "confidence": round(score.item(), 3),
        "bounding_box": [round(i, 2) for i in box.tolist()]
      })

    return detected_tables

# Initialize Model

In [ ]:
detector = TableDetection("./config.yaml")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

The image processor of type `DetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Initialize API

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


## TODO
@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    # Đọc dữ liệu binary của ảnh
    imageBytes = await file.read()
    # Gọi model xử lý
    results = detector(imageBytes)
    return {
        "filename": file.filename,
        "detected_objects": results
    }
## END TODO

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")

Server started on http://0.0.0.0:8000


# Call Local API

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/predict"

with open("test4.jpg", "rb") as image_file:
    files = {"file": image_file}
    response = requests.post(API_URL, files=files)

print(response.json())

INFO:     127.0.0.1:47630 - "POST /predict HTTP/1.1" 200 OK
{'filename': 'test4.jpg', 'detected_objects': []}
